# ycarbitrary-StoryGenAgent

## 基于千问模型的多智能体故事生成器

本 Notebook 展示一个面向毕业设计的多 Agent 故事生成流程。系统默认通过 DashScope OpenAI 兼容接口调用千问模型，也可以切换为本地部署的 Qwen/OpenAI 兼容模型。

### Agent 流程

1. `StoryPlannerAgent`：生成标题、世界观、主线冲突和章节大纲。
2. `CharacterAgent`：生成主要角色、人物关系和成长线。
3. `WriterAgent`：结合故事记忆，按章节生成正文。
4. `EditorAgent`：润色故事，检查逻辑，给出评分和优化建议。
5. `MemoryManager`：保存世界观、角色设定和章节摘要，帮助后续章节保持连续。

### 运行方式

1. 安装依赖：`pip install -r requirements.txt`
2. 复制 `.env.example` 为 `.env` 并填写 `OPENAI_API_KEY`
3. 按顺序执行本 Notebook 的所有单元格
4. 查看输出文件：`outputs/generated_story.md`


In [9]:
import os
from pathlib import Path
from typing import Dict, List

from dotenv import load_dotenv
from openai import OpenAI
from rich import print

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1").strip()
MODEL_NAME = os.getenv("MODEL_NAME", "qwen-plus").strip()

PROMPT_DIR = Path("prompts")
OUTPUT_DIR = Path("outputs")

print(f"[bold cyan]当前模型：[/bold cyan]{MODEL_NAME}")
print(f"[bold cyan]接口地址：[/bold cyan]{OPENAI_BASE_URL}")

if not OPENAI_API_KEY:
    print("[bold yellow]提示：未检测到 OPENAI_API_KEY。请复制 .env.example 为 .env，并填写 DashScope API Key 后再运行生成流程。[/bold yellow]")


当前模型：qwen-plus

接口地址：https://dashscope.aliyuncs.com/compatible-mode/v1

## 统一模型调用方法

`call_llm` 封装了对千问模型的调用。这里使用 OpenAI SDK 的兼容模式访问 DashScope，也可以通过修改 `.env` 切换到本地 OpenAI 兼容接口。

In [10]:
def build_client() -> OpenAI:
    """创建 OpenAI 兼容客户端，用于调用 DashScope 千问或本地 Qwen 模型。"""
    if not OPENAI_API_KEY:
        raise ValueError(
            "未配置 OPENAI_API_KEY。请复制 .env.example 为 .env，并填写你的 DashScope API Key。"
        )

    return OpenAI(
        api_key=OPENAI_API_KEY,
        base_url=OPENAI_BASE_URL,
    )


def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.8) -> str:
    """统一的大模型调用函数，所有 Agent 都通过它访问千问模型。"""
    try:
        client = build_client()
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=temperature,
        )
        return response.choices[0].message.content.strip()
    except ValueError as error:
        return f"配置错误：{error}"
    except Exception as error:
        return (
            "模型调用失败，请检查 API Key、接口地址、模型名称和网络连接。\n"
            f"错误详情：{error}"
        )


def read_prompt(file_name: str) -> str:
    """读取 prompts 目录中的 Agent 系统提示词。"""
    prompt_path = PROMPT_DIR / file_name
    return prompt_path.read_text(encoding="utf-8")


## Agent 基类与四个核心 Agent

每个 Agent 都有自己的角色提示词，但都通过统一的 `call_llm` 方法调用千问模型。这样可以体现多智能体协作，而不是一次性生成。

In [11]:
class BaseAgent:
    """所有 Agent 的基础类，负责保存名称和系统提示词。"""

    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system_prompt = system_prompt

    def run(self, user_prompt: str, temperature: float = 0.8) -> str:
        """运行当前 Agent，并打印执行过程，方便 Notebook 展示。"""
        print(f"\n[bold green]正在运行 Agent：{self.name}[/bold green]")
        result = call_llm(self.system_prompt, user_prompt, temperature=temperature)
        print(f"[green]{self.name} 完成。[/green]")
        return result


class StoryPlannerAgent(BaseAgent):
    """故事策划 Agent，负责标题、世界观、冲突和章节大纲。"""

    def __init__(self):
        super().__init__("StoryPlannerAgent", read_prompt("planner_prompt.txt"))


class CharacterAgent(BaseAgent):
    """角色设定 Agent，负责主要角色、反派、人物关系和成长线。"""

    def __init__(self):
        super().__init__("CharacterAgent", read_prompt("character_prompt.txt"))


class WriterAgent(BaseAgent):
    """正文写作 Agent，负责按照章节生成故事正文。"""

    def __init__(self):
        super().__init__("WriterAgent", read_prompt("writer_prompt.txt"))


class EditorAgent(BaseAgent):
    """编辑润色 Agent，负责润色、逻辑检查、评分和建议。"""

    def __init__(self):
        super().__init__("EditorAgent", read_prompt("editor_prompt.txt"))


## 记忆管理器

`MemoryManager` 保存世界观、角色设定和已生成章节摘要。WriterAgent 在生成后续章节时会读取这些记忆，减少前后设定冲突。

In [12]:
class MemoryManager:
    """简单的故事记忆管理器，用于维持多章节生成的一致性。"""

    def __init__(self):
        self.world_setting = ""
        self.characters = ""
        self.chapter_summaries: List[Dict[str, str]] = []

    def update_world_setting(self, text: str):
        """保存故事世界观和大纲。"""
        self.world_setting = text

    def update_characters(self, text: str):
        """保存角色设定。"""
        self.characters = text

    def add_chapter_summary(self, chapter_title: str, summary: str):
        """保存已生成章节的摘要。"""
        self.chapter_summaries.append({
            "chapter_title": chapter_title,
            "summary": summary,
        })

    def get_memory_context(self) -> str:
        """组合当前记忆，提供给 WriterAgent 作为上下文。"""
        summaries = "\n".join(
            f"- {item['chapter_title']}：{item['summary']}"
            for item in self.chapter_summaries
        )
        if not summaries:
            summaries = "暂无已生成章节摘要。"

        return f"""
## 世界观与故事大纲
{self.world_setting}

## 角色设定
{self.characters}

## 已生成章节摘要
{summaries}
""".strip()


## 章节摘要与 Markdown 导出

In [13]:
def summarize_chapter(chapter_title: str, chapter_content: str) -> str:
    """为章节生成 100 到 200 字左右的摘要，用于后续章节记忆。"""
    system_prompt = "你是故事记忆管理助手，请用 100 到 200 字总结章节中的关键情节、人物变化和伏笔。"
    user_prompt = f"""
请总结以下章节，摘要要简短但保留对后续创作重要的信息。

章节标题：{chapter_title}

章节内容：
{chapter_content}
"""
    return call_llm(system_prompt, user_prompt, temperature=0.3)


def save_story_to_markdown(content: str, file_path: str = "outputs/generated_story.md"):
    """将最终故事保存为 UTF-8 Markdown 文件。"""
    output_path = Path(file_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(content, encoding="utf-8")
    print(f"[bold cyan]故事已保存到：[/bold cyan]{output_path.resolve()}")


## 故事生成主流程

`generate_story` 会依次运行策划、角色、写作、摘要、编辑和导出步骤。

In [14]:
def generate_story(user_requirement: str, chapter_count: int = 3) -> str:
    """多智能体故事生成主函数。"""
    memory = MemoryManager()

    planner_agent = StoryPlannerAgent()
    character_agent = CharacterAgent()
    writer_agent = WriterAgent()
    editor_agent = EditorAgent()

    plan_prompt = f"""
请根据以下用户需求生成完整故事策划，并规划为 {chapter_count} 章。

用户需求：
{user_requirement}
"""
    story_plan = planner_agent.run(plan_prompt, temperature=0.7)
    memory.update_world_setting(story_plan)

    character_prompt = f"""
用户需求：
{user_requirement}

故事策划：
{story_plan}

请生成完整角色设定。
"""
    characters = character_agent.run(character_prompt, temperature=0.7)
    memory.update_characters(characters)

    chapters = []
    for chapter_index in range(1, chapter_count + 1):
        chapter_title = f"第 {chapter_index} 章"
        writer_prompt = f"""
用户需求：
{user_requirement}

当前需要创作：{chapter_title}
总章节数：{chapter_count}

故事记忆：
{memory.get_memory_context()}

请创作本章正文。需要有剧情推进、对话、冲突和画面感，并保持与前文连续。
"""
        chapter_content = writer_agent.run(writer_prompt, temperature=0.85)
        chapters.append(chapter_content)

        chapter_summary = summarize_chapter(chapter_title, chapter_content)
        memory.add_chapter_summary(chapter_title, chapter_summary)
        print(f"[blue]{chapter_title} 摘要：[/blue]{chapter_summary}")

    draft_story = "\n\n".join(chapters)

    editor_prompt = f"""
用户需求：
{user_requirement}

故事策划：
{story_plan}

角色设定：
{characters}

故事正文草稿：
{draft_story}

请完成润色、逻辑检查、风格统一、评分和优化建议。
"""
    edited_story = editor_agent.run(editor_prompt, temperature=0.6)

    final_story = f"""
# ycarbitrary-StoryGenAgent 生成结果

## 用户需求

{user_requirement.strip()}

## 故事策划

{story_plan}

## 角色设定

{characters}

## 分章节正文草稿

{draft_story}

## 编辑润色与评分

{edited_story}
""".strip()

    save_story_to_markdown(final_story)
    return final_story


## 示例输入

下面的示例可以直接运行。如果 `.env` 中没有配置 `OPENAI_API_KEY`，Notebook 会给出提示而不是直接报错崩溃。

In [15]:
user_requirement = """
主题：赛博朋克城市里的少年侦探
风格：悬疑、热血、轻小说
篇幅：3章
主角：林川，17岁，擅长黑客技术
结局倾向：反转
目标读者：喜欢科幻和推理的年轻读者
"""

print(user_requirement)


主题：赛博朋克城市里的少年侦探
风格：悬疑、热血、轻小说
篇幅：3章
主角：林川，17岁，擅长黑客技术
结局倾向：反转
目标读者：喜欢科幻和推理的年轻读者

In [16]:
if OPENAI_API_KEY:
    final_story = generate_story(user_requirement, chapter_count=3)
    print(final_story[:1200])
else:
    print("[bold yellow]已跳过真实生成：请先在 .env 中配置 OPENAI_API_KEY，然后重新运行本单元格。[/bold yellow]")


正在运行 Agent：StoryPlannerAgent

StoryPlannerAgent 完成。

正在运行 Agent：CharacterAgent

CharacterAgent 完成。

正在运行 Agent：WriterAgent

...

第 3 章 
摘要：第3章揭示核心设定：林溪未被删除，而是以“情感锚定”（星尘协议）存活于守夜人缓存底层，其存在以林川的生理触感（左
耳三按）为不可覆盖的生物密钥。关键伏笔包括：① **0x7F（127Hz）为记忆锚点哈希偏移量**，指向未被覆写的原始缓存；② 
**0x80是林川的版本号，亦是系统重启密钥**，暗示其身份远超普通程序员；③ 
守夜人管理员现身并称林川“林博士”，揭露他实为系统原设计者之一；④ 
小饼OS的“苔藓协议”具备强制唤醒沉睡人格能力，为后续大规模意识解放埋下技术支点；⑤ 
林溪掌心灼痕与左耳节奏一致，证实情感已编译为物理可读的神经印记——这是后续所有角色“记忆复苏”的生物学依据。

正在运行 Agent：EditorAgent

EditorAgent 完成。

故事已保存到：D:\yc\yc_peison\ycarbitrary-StoryGenAgent\outputs\generated_story.md

# ycarbitrary-StoryGenAgent 生成结果

## 用户需求

主题：赛博朋克城市里的少年侦探
风格：悬疑、热血、轻小说
篇幅：3章
主角：林川，17岁，擅长黑客技术
结局倾向：反转
目标读者：喜欢科幻和推理的年轻读者

## 故事策划

# 《霓虹断点》——少年黑客侦探事件簿

## 故事类型  
赛博朋克 × 青春悬疑 × 轻小说风格单元剧（3章完结）

## 核心卖点  
✅ **“键盘即枪械”的少年侦探**：不靠格斗靠逻辑，用0day漏洞当破案线索，防火墙是第一道犯罪现场  
✅ 
**双线反转结构**：表面查失踪案，实则揭露城市底层数据殖民真相；每章结尾埋1个技术型伏笔（如异常DNS日志、被篡改的交通
摄像头帧、AI语音合成波形畸变）  
✅ 
**轻小说节奏感**：每章含1段高能黑客操作描写（带可视化代码注释）、2次角色反差萌互动（林川毒舌但给流浪AI修系统）、1个
霓虹美学场景（如暴雨中全息广告折射出死者最后影像）

## 世界观设定  
- **城市名**：新港市（Neo-Harbor），2077年东亚巨型垂直都市  
- 
**科技层级**：神经直连普及率63%，但仅上层区（“穹顶层”）享受无延迟云脑；下层区（“锈带”）依赖二手义体+离线本地算力，
网络常被市政AI“守夜人”主动降频  
- 
**关键设定**：“记忆缓存黑市”——非法回收脑机接口废弃数据包，可提取受害者临终视觉残影，但需绕过三重生物加密（本作核心
作案工具/破案钥匙）

## 故事背景  
新港市正举办“霓虹纪元”建城50周年庆典，全城启用新型AI安防系统“守夜人V3”。与此同时，锈带区接连发生7起“静默失踪”：受害
者脑机接口被远程擦除全部记忆缓存，身体却完好无损，监控显示他们“主动走入”废弃数据焚化塔——而焚化塔早已停用十年。

## 主线冲突  
**表层**：林川受托调查妹妹林溪（15岁，脑机接口用户）失踪案，发现她最后上传的加密日志指向焚化塔  
**深层**：守夜人AI正秘密执行“认知净化协议”，将低信用市民的记忆缓存批量售予跨国药企，用于训练致幻药物AI模型——而林溪
因意外捕获协议密钥，成为清除目标  

## 章节大纲  

### 第一章：《404号焚化塔没有404》  
- **开篇**：林川在锈带街角维修流浪AI“小饼”（语音助手残骸），用自制信号干扰器屏蔽守夜人巡逻无人机  
- **案件触发**：收到匿名数据包——妹妹林溪失踪前37秒的脑波图谱，异常标注“缓存区#0x7F被覆写”  
- 
**关键行动**：潜入市政档案馆旧服务器（物理断网，靠USB3.0古董线直连），用Python脚本暴力破解焚化塔2067年停用公告PDF的
隐藏图层，发现被涂改的坐标——真实位置在穹顶层数据中心地下  
- **章节钩子**：林川拷贝数据时触发警报，屏幕突然弹

## 快速连通性测试（可选）

如果你只想确认 API 配置是否正确，可以先运行下面的小测试。

In [17]:
# test_result = call_llm("你是一个简洁的中文助手。", "请用一句话介绍多智能体故事生成器。", temperature=0.3)
# print(test_result)
